# 541 - FAISS: Product Vector Search

Build a small product search system in two parts:

1. Search numeric vectors made from price, capacity, and weight. No embedding model or cloud account is needed.
2. Create text embeddings with Google, search by meaning, filter results, and save files for a later RAG notebook.

FAISS is a vector similarity search library. It stores vectors and searches them; it does not store product records or provide a complete database service. Keep the product metadata alongside the index.

All 20 products and brands below are fictional. Prices are in INR. Run one cell at a time, from top to bottom.

## Install the required libraries

Use the existing notebook environment. Run the cell below to install only the libraries used in this notebook. FAISS runs on the CPU; no CUDA or GPU is required. Text embeddings in Part B use Google Cloud.

In [ ]:
%pip install -q "faiss-cpu==1.13.2" "numpy>=1.26,<3" "google-genai==2.25.0"

## 1. Import the libraries

Restart the kernel after installation if prompted, then continue here. Use `float32` arrays for FAISS.

In [ ]:
import json
from pathlib import Path
import numpy as np
import faiss

print(
    f"FAISS version: {faiss.__version__}\n"
    f"NumPy version: {np.__version__}\n"
    "Search device: CPU"
)

## 2. Stage 20 products

Use one product family so measurements are comparable. Backpacks have a capacity in litres, a weight in grams, and dimensions in centimetres. Brand and intended use are metadata; brand names are not numeric measurements.

Read `data/products.json`, the shared catalogue used by 542 and 543. Product IDs link FAISS matches to MySQL order items. Current catalogue prices can differ from historical order-item sale prices.

In [ ]:
products = json.loads(Path("data/products.json").read_text(encoding="utf-8"))
assert len(products) == 20
assert len({p["product_id"] for p in products}) == 20
print(
    f"Shared products staged: {len(products)}\n"
    f"Categories: {sorted({p['category'] for p in products})}\n"
    f"First product:\n{json.dumps(products[0], indent=2)}"
)

## 3. Print the catalogue

Keep each product's ID with its metadata. FAISS row position 0 will refer to `products[0]`, position 1 to `products[1]`, and so on.

In [ ]:
for p in products:
    print(
        f"{p['product_id']} | {p['name']} | {p['brand']}\n"
        f"  Category: {p['category']} | Price: INR {p['price_inr']}\n"
        f"  Capacity: {p['capacity_l']} L | Weight: {p['weight_g']} g\n"
        f"  Size (H x W x D): {p['height_cm']} x {p['width_cm']} x {p['depth_cm']} cm\n"
    )

## Part A - Search without an embedding model

### 4. Turn measurements into three-number vectors

Use `[price, capacity, weight]`. Divide by fixed scales so a price difference does not overwhelm a capacity difference simply because their units differ.

Here INR 1000, 10 litres, and 500 grams each become one unit of distance. These are chosen comparison weights, not learned values. Use the same scales for the products and query. Dimensions and brand remain metadata in this example.

In [ ]:
feature_names = ["price_inr", "capacity_l", "weight_g"]
feature_scales = np.array([1000, 10, 500], dtype="float32")
raw_vectors = np.array(
    [[p[name] for name in feature_names] for p in products], dtype="float32"
)
numeric_vectors = np.ascontiguousarray(raw_vectors / feature_scales)
print(
    f"Feature order: {feature_names}\n"
    f"Array shape: {numeric_vectors.shape} (products, features)\n"
    f"First raw vector: {raw_vectors[0]}\n"
    f"First scaled vector: {numeric_vectors[0]}"
)

### 5. Create an index and add the vectors

`IndexFlatL2` compares every stored vector using squared Euclidean distance. Smaller scores mean closer measurements. This exact index needs no training.

In [ ]:
numeric_index = faiss.IndexFlatL2(numeric_vectors.shape[1])
numeric_index.add(numeric_vectors)
print(
    f"Vector dimensions: {numeric_index.d}\n"
    f"Stored vectors: {numeric_index.ntotal}\n"
    f"Ready to search: {numeric_index.is_trained}"
)

### 6. Search for a similar size and price

Find five products close to INR 2500, 25 litres, and 700 grams. This is a preference, not a hard maximum price. A result can cost more than INR 2500.

In [ ]:
wanted = np.array([[2500, 25, 700]], dtype="float32")
numeric_query = np.ascontiguousarray(wanted / feature_scales)
distances, positions = numeric_index.search(numeric_query, k=5)
print(
    f"Query [INR, litres, grams]: {wanted[0]}\n"
    f"Returned row positions: {positions[0]}\n"
    f"Squared distances: {distances[0]}\n"
)
for rank, (position, distance) in enumerate(zip(positions[0], distances[0]), start=1):
    p = products[int(position)]
    print(
        f"{rank}. {p['product_id']} | {p['name']}\n"
        f"   INR {p['price_inr']} | {p['capacity_l']} L | {p['weight_g']} g\n"
        f"   Squared distance: {distance:.4f}\n"
    )

### 7. Check one distance by hand

Subtract the query from the best match, square each difference, and add them. FAISS returns the squared distance, without taking a square root.

In [ ]:
best_position = int(positions[0, 0])
difference = numeric_vectors[best_position] - numeric_query[0]
manual_distance = np.sum(difference ** 2)
print(
    f"Difference: {difference}\n"
    f"Squared differences: {difference ** 2}\n"
    f"Manual total: {manual_distance:.4f}\n"
    f"FAISS distance: {distances[0, 0]:.4f}"
)
assert np.isclose(manual_distance, distances[0, 0])

### 8. Save and reload the numeric index

The index alone cannot display product names. Save its metadata and feature scales too. Files go under `output/541_faiss/numeric/` relative to this notebook's working directory. Rerunning replaces this example's files.

In [ ]:
artifact_root = Path("output/541_faiss")
numeric_dir = artifact_root / "numeric"
numeric_dir.mkdir(parents=True, exist_ok=True)
faiss.write_index(numeric_index, str(numeric_dir / "products.faiss"))
(numeric_dir / "products.json").write_text(json.dumps(products, indent=2), encoding="utf-8")
(numeric_dir / "features.json").write_text(
    json.dumps({"names": feature_names, "scales": feature_scales.tolist()}, indent=2),
    encoding="utf-8",
)
reloaded_numeric = faiss.read_index(str(numeric_dir / "products.faiss"))
reloaded_distances, reloaded_positions = reloaded_numeric.search(numeric_query, k=5)
assert np.array_equal(positions, reloaded_positions)
assert np.allclose(distances, reloaded_distances)
print(
    f"Saved to: {numeric_dir.resolve()}\n"
    f"Reloaded vectors: {reloaded_numeric.ntotal}\n"
    "Search results match after reloading."
)

### 8a. What is cosine similarity?

Think of a vector as an arrow. **Cosine similarity compares the directions of two arrows**, rather than their lengths. It is the cosine of the angle between them:

$$\operatorname{cosine}(a,b)=\frac{a\cdot b}{\|a\|\,\|b\|}$$

- `a dot b`: multiply matching coordinates and add the results.
- `||a||`: the length of vector `a`, calculated as the square root of the sum of squared coordinates.
- `1`: same direction, even if one arrow is longer.
- `0`: perpendicular directions.
- `-1`: opposite directions.

A zero-length vector has no direction, so its cosine similarity is undefined. For text embeddings, higher cosine similarity is often useful for finding related text, but it is not a probability or a guarantee of relevance.

Start with small, hand-written vectors. These are geometric examples, not model embeddings.

In [ ]:
cosine_query = np.array([1, 0], dtype="float32")
example_names = ["Same direction", "Longer, same direction", "Diagonal", "Perpendicular", "Opposite"]
example_vectors = np.array([
    [1, 0],
    [2, 0],
    [1, 1],
    [0, 1],
    [-1, 0],
], dtype="float32")

manual_cosines = []
for name, vector in zip(example_names, example_vectors):
    dot_product = np.dot(cosine_query, vector)
    query_length = np.linalg.norm(cosine_query)
    vector_length = np.linalg.norm(vector)
    cosine = dot_product / (query_length * vector_length)
    manual_cosines.append(float(cosine))
    print(
        f"{name}: {vector}\n"
        f"  Dot product: {dot_product:.4f}\n"
        f"  Lengths: {query_length:.4f} and {vector_length:.4f}\n"
        f"  Cosine similarity: {cosine:.4f}\n"
    )

### 8b. Search the same vectors with FAISS

Normalize each vector by dividing it by its length. This keeps its direction and makes its length equal to 1. Now the dot product alone equals cosine similarity, because the denominator is `1 x 1`.

FAISS calls the dot product an **inner product**. Use `IndexFlatIP` with normalized stored vectors and a normalized query. Search returns the largest scores first. The two same-direction examples tie at 1; their relative order is not important.

Cosine ignores magnitude: `[1, 0]` and `[2, 0]` both score 1. That is why Part A uses distance to compare actual price and size, while Part B uses cosine to compare text embedding directions.

In [ ]:
unit_examples = example_vectors.copy()
unit_query = cosine_query.reshape(1, -1).copy()
faiss.normalize_L2(unit_examples)
faiss.normalize_L2(unit_query)

cosine_demo_index = faiss.IndexFlatIP(2)
cosine_demo_index.add(unit_examples)
demo_scores, demo_positions = cosine_demo_index.search(unit_query, k=len(example_names))
print(
    f"Normalized vector lengths: {np.linalg.norm(unit_examples, axis=1)}\n"
    "FAISS cosine search results:\n"
)
for position, score in zip(demo_positions[0], demo_scores[0]):
    print(
        f"{example_names[int(position)]}\n"
        f"  Manual cosine: {manual_cosines[int(position)]:.4f}\n"
        f"  FAISS score: {score:.4f}\n"
    )
assert np.allclose(demo_scores[0], np.array(manual_cosines)[demo_positions[0]])

## Part B - Search using text embeddings

### 9. Create the embedding client

Use the existing Google Cloud configuration and Application Default Credentials (ADC). The client reads the shared project, embedding model, and location from `course_config.py`.

The embedding section makes 20 document calls and two query calls. Rerunning these cells repeats the requests.

In [ ]:
from google.genai import types
from course_config import PROJECT_ID, EMBEDDING_MODEL, EMBEDDING_LOCATION, make_client

EMBEDDING_DIMENSIONS = 768
embedding_client = make_client(location=EMBEDDING_LOCATION)
print(
    f"Project: {PROJECT_ID}\n"
    f"Embedding model: {EMBEDDING_MODEL}\n"
    f"Embedding location: {EMBEDDING_LOCATION}\n"
    f"Requested dimensions: {EMBEDDING_DIMENSIONS}"
)

### 10. Create one document per product

Keep short product records whole. Each document includes a stable product ID, description, and measurements. These same documents will become source context for RAG later.

In [ ]:
documents = []
for p in products:
    document = (
        f"Product ID: {p['product_id']}\n"
        f"Name: {p['name']}\n"
        f"Brand: {p['brand']}\n"
        f"Category: {p['category']}\n"
        f"Price: INR {p['price_inr']}\n"
        f"Capacity: {p['capacity_l']} litres\n"
        f"Weight: {p['weight_g']} grams\n"
        f"Dimensions (H x W x D): {p['height_cm']} x {p['width_cm']} x {p['depth_cm']} cm\n"
        f"Description: {p['description']}"
    )
    documents.append(document)
print(
    f"Documents prepared: {len(documents)}\n\n"
    f"Example document:\n{documents[5]}"
)

### 11. Embed the documents

An embedding model converts text into a vector that can capture meaning. Unlike Part A, individual dimensions are not named measurements.

Use `RETRIEVAL_DOCUMENT` for stored products and `RETRIEVAL_QUERY` for searches. Both must use the same model and output dimension. Send one product per request for this small example.

In [ ]:
def embed_text(text, task_type):
    response = embedding_client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=text,
        config=types.EmbedContentConfig(
            task_type=task_type,
            output_dimensionality=EMBEDDING_DIMENSIONS,
        ),
    )
    return np.asarray(response.embeddings[0].values, dtype="float32")

embedding_rows = []
for p, document in zip(products, documents):
    embedding_rows.append(embed_text(document, "RETRIEVAL_DOCUMENT"))
    print(f"Embedded: {p['product_id']} | {p['name']}")

product_embeddings = np.ascontiguousarray(np.vstack(embedding_rows), dtype="float32")
assert product_embeddings.shape == (len(products), EMBEDDING_DIMENSIONS)
assert np.isfinite(product_embeddings).all()
assert np.all(np.linalg.norm(product_embeddings, axis=1) > 0)
print(
    f"\nEmbedding shape: {product_embeddings.shape}\n"
    f"First vector, first 8 values: {product_embeddings[0, :8]}"
)

### 12. Normalize and index the embeddings

Apply the same steps from the small cosine example to the product embeddings: normalize every vector to length 1, then use `IndexFlatIP` (inner product). The resulting scores are cosine similarities: larger means more similar. They are not probabilities or guarantees of relevance.

Normalize query vectors too. Numeric L2 distances from Part A and cosine scores from Part B use different scales and cannot be compared directly.

In [ ]:
faiss.normalize_L2(product_embeddings)
semantic_index = faiss.IndexFlatIP(EMBEDDING_DIMENSIONS)
semantic_index.add(product_embeddings)
print(
    f"Index dimensions: {semantic_index.d}\n"
    f"Indexed products: {semantic_index.ntotal}\n"
    f"First vector length: {np.linalg.norm(product_embeddings[0]):.4f}"
)

### 13. Search by meaning

Search with ordinary language rather than a three-number vector. Inspect the returned descriptions: similarity can miss negation, such as 'not waterproof', and does not enforce exact numeric requirements.

In [ ]:
query_text = "A waterproof backpack for hiking in the rain with comfortable support"
query_vector = np.ascontiguousarray(
    embed_text(query_text, "RETRIEVAL_QUERY").reshape(1, -1)
)
faiss.normalize_L2(query_vector)
scores, semantic_positions = semantic_index.search(query_vector, k=5)
print(f"Query:\n{query_text}\n")
for rank, (position, score) in enumerate(zip(semantic_positions[0], scores[0]), start=1):
    p = products[int(position)]
    print(
        f"{rank}. {p['product_id']} | {p['name']} | Similarity: {score:.4f}\n"
        f"   Brand: {p['brand']} | INR {p['price_inr']}\n"
        f"   {p['description']}\n"
    )

### 14. Apply exact metadata filters

Require category `hiking`, brand `PeakPath`, price at most INR 3000, and weight at most 1000 grams. A vector score alone cannot enforce these conditions.

For only 20 products, retrieve all ranked rows, then apply the filters and take the best three eligible matches. Filtering only the top three unfiltered results could miss eligible products. For a large catalogue, use a filtered candidate index or a vector database with filtering support.

In [ ]:
max_price = 3000
max_weight = 1000
required_brand = "PeakPath"
all_scores, all_positions = semantic_index.search(query_vector, k=semantic_index.ntotal)
filtered_results = []
for position, score in zip(all_positions[0], all_scores[0]):
    p = products[int(position)]
    if (
        p["category"] == "hiking"
        and p["brand"] == required_brand
        and p["price_inr"] <= max_price
        and p["weight_g"] <= max_weight
    ):
        filtered_results.append({"row": int(position), "score": float(score), "product": p})
filtered_results = filtered_results[:3]
print(f"Eligible matches returned: {len(filtered_results)}\n")
for match in filtered_results:
    p = match["product"]
    print(
        f"{p['product_id']} | {p['name']} | {p['brand']}\n"
        f"Price: INR {p['price_inr']} | Weight: {p['weight_g']} g\n"
        f"Similarity: {match['score']:.4f}\n"
    )
if not filtered_results:
    print("No products satisfy these filters. Change the filters and try again.")

### 15. Try another query

Try 'a small bag for a child', 'a bag that opens like a suitcase', or the laptop query below. Compare the returned descriptions instead of assuming the first result is correct.

In [ ]:
second_query = "A slim laptop backpack for travelling to the office by train"
second_vector = np.ascontiguousarray(
    embed_text(second_query, "RETRIEVAL_QUERY").reshape(1, -1)
)
faiss.normalize_L2(second_vector)
second_scores, second_positions = semantic_index.search(second_vector, k=3)
print(f"Query:\n{second_query}\n")
for position, score in zip(second_positions[0], second_scores[0]):
    p = products[int(position)]
    print(
        f"{p['product_id']} | {p['name']} | Similarity: {score:.4f}\n"
        f"{p['description']}\n"
    )

### 16. Save the retrieval bundle for the next notebook

Save the index, ordered product records, source documents, normalized vectors, model configuration, and one example query vector. Keep these files together. Changing the embedding model, dimension, document text, or product order requires rebuilding the bundle.

The numeric and semantic indexes use separate directories so they cannot be confused.

In [ ]:
import hashlib

semantic_dir = artifact_root / "semantic"
semantic_dir.mkdir(parents=True, exist_ok=True)
records = [dict(p, document=doc) for p, doc in zip(products, documents)]
manifest = {
    "schema_version": 1,
    "catalogue_sha256": hashlib.sha256(json.dumps(products, sort_keys=True, separators=(",", ":")).encode("utf-8")).hexdigest(),
    "embedding_model": EMBEDDING_MODEL,
    "embedding_location": EMBEDDING_LOCATION,
    "dimensions": EMBEDDING_DIMENSIONS,
    "document_task": "RETRIEVAL_DOCUMENT",
    "query_task": "RETRIEVAL_QUERY",
    "normalized": True,
    "metric": "cosine_via_inner_product",
    "index_type": "IndexFlatIP",
    "product_count": len(records),
    "row_product_ids": [p["product_id"] for p in records],
    "faiss_version": faiss.__version__,
}
faiss.write_index(semantic_index, str(semantic_dir / "products.faiss"))
np.save(semantic_dir / "product_vectors.npy", product_embeddings, allow_pickle=False)
np.save(semantic_dir / "example_query.npy", query_vector, allow_pickle=False)
(semantic_dir / "products.json").write_text(json.dumps(records, indent=2), encoding="utf-8")
(semantic_dir / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
(semantic_dir / "example_query.json").write_text(
    json.dumps({"text": query_text}, indent=2), encoding="utf-8"
)
print(
    f"Saved bundle: {semantic_dir.resolve()}\n"
    "products.faiss: vector index\n"
    "products.json: ordered metadata and source documents\n"
    "manifest.json: model, dimensions, metric and row IDs\n"
    "product_vectors.npy: normalized document vectors\n"
    "example_query.npy / example_query.json: replayable search"
)

### 17. Reload and replay without another embedding call

This cell loads its own files and can run in a fresh kernel after the imports in section 1. Only load index files you created or trust. Verify dimensions, row IDs, and counts before attaching metadata to search results.

The saved query vector can replay this particular search offline. A new question still needs an embedding from the same model and configuration.

In [ ]:
bundle_dir = Path("output/541_faiss/semantic")
loaded_manifest = json.loads((bundle_dir / "manifest.json").read_text(encoding="utf-8"))
loaded_products = json.loads((bundle_dir / "products.json").read_text(encoding="utf-8"))
loaded_index = faiss.read_index(str(bundle_dir / "products.faiss"))
loaded_query = np.load(bundle_dir / "example_query.npy", allow_pickle=False)
loaded_question = json.loads((bundle_dir / "example_query.json").read_text(encoding="utf-8"))["text"]
assert loaded_manifest["schema_version"] == 1
assert loaded_manifest["metric"] == "cosine_via_inner_product"
assert loaded_manifest["normalized"] is True
assert loaded_index.metric_type == faiss.METRIC_INNER_PRODUCT
assert loaded_index.ntotal == len(loaded_products) == loaded_manifest["product_count"]
assert loaded_index.d == loaded_manifest["dimensions"] == loaded_query.shape[1]
assert [p["product_id"] for p in loaded_products] == loaded_manifest["row_product_ids"]
replay_scores, replay_positions = loaded_index.search(loaded_query, k=min(3, loaded_index.ntotal))
print(
    f"Reloaded products: {loaded_index.ntotal}\n"
    f"Saved question: {loaded_question}\n"
    f"Top product IDs: {[loaded_products[int(i)]['product_id'] for i in replay_positions[0]]}"
)

### 18. Prepare context for a later RAG notebook

Retrieval returns relevant source records. A separate RAG notebook can give a question and these records to a generation model, ask for an answer grounded in the records, and cite the product IDs. No answer is generated here.

Use the reloaded records to prepare context. If a question has hard price, brand, or size requirements, apply those metadata filters before building context, as in section 14. These replay results demonstrate an unfiltered semantic search.

In [ ]:
context_blocks = []
retrieved_sources = []
for position, score in zip(replay_positions[0], replay_scores[0]):
    p = loaded_products[int(position)]
    context_blocks.append(f"[{p['product_id']}]\n{p['document']}")
    retrieved_sources.append({"product_id": p["product_id"], "score": float(score)})
rag_context = "\n\n".join(context_blocks)
(bundle_dir / "example_context.txt").write_text(rag_context, encoding="utf-8")
(bundle_dir / "example_retrieval.json").write_text(
    json.dumps({"question": loaded_question, "sources": retrieved_sources}, indent=2),
    encoding="utf-8",
)
print(
    "Retrieved context for the next notebook:\n\n"
    f"{rag_context}\n\n"
    "Next step: load this bundle, embed a new question, retrieve records, then generate a cited answer."
)

## Try it

- Change the numeric query and explain why the closest product changes.
- Change one feature scale. Which measurement now matters more?
- Set `max_price = 100` in section 14 and confirm that no results are returned.
- Change the cosine query to `[0, 1]` and predict the ranking before searching.
- Compare the descriptions for a hiking query and a laptop query.
- Add a product, rebuild the vectors and index, and save a new consistent bundle.

## References

- [FAISS getting started](https://github.com/facebookresearch/faiss/wiki/Getting-started)
- [FAISS distance metrics](https://github.com/facebookresearch/faiss/wiki/MetricType-and-distances)
- [CPU package](https://pypi.org/project/faiss-cpu/1.13.2/)
- [Google text embeddings](https://docs.cloud.google.com/vertex-ai/generative-ai/docs/embeddings/get-text-embeddings)

For the next notebook's file contract, see [541_RAG_HANDOFF.md](541_RAG_HANDOFF.md).

Next: 542 links these product IDs to orders and order items. 543 lets an agent combine semantic matches with MySQL facts and pandas calculations.